# Thinking probe + pruning on Qwen3-0.6B

Runtime → Change runtime type → **T4 GPU** (or better). Run the cells in order.

On a T4 keep `DTYPE = "float16"`; on an L4/A100 use `"bfloat16"`.

In [ ]:
DTYPE = "float16"   # "bfloat16" on L4 / A100
PER_SOURCE = 25      # questions per dataset (4 datasets); raise for the real run
!nvidia-smi

In [ ]:
!git clone -q https://github.com/BottleCapAI/ThinkingCap-Test
%cd ThinkingCap-Test
!pip -q install uv && uv pip install --system -q -r requirements.txt

Upload `probe_curve.py`, `prune_traces.py` and `sft.py` (the files Claude sent you).

In [ ]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded))

In [ ]:
!python utils/train/prepare_data.py

## 1. How early does the model know the answer?
Samples 4 answers per question, then reads p(correct) every 16 thinking tokens.

In [ ]:
!python probe_curve.py --out runs/probe --per-source $PER_SOURCE --samples 4 --dtype $DTYPE

In [ ]:
from IPython.display import Image, display
print(open('runs/probe/summary.txt').read())
display(Image('runs/probe/curve.png'))

## 2. Build shorter training traces (cut after the answer settles, delete steps that add nothing)

In [ ]:
!python prune_traces.py --rollouts runs/probe/rollouts.jsonl --out runs/prune --dtype $DTYPE
print(open('runs/prune/summary.txt').read())

## 3. (Optional) Fine-tune on them
Only meaningful with more data: rerun step 1 on `data/train.jsonl` with a few hundred questions per dataset first (`--data data/train.jsonl`).

In [ ]:
!python sft.py --data runs/prune/sft.jsonl --out adapters/pruned-sft --dtype $DTYPE

## Send back to Claude
Download these and share them in the chat.

In [ ]:
!zip -q -j results.zip runs/probe/summary.txt runs/probe/curve.png runs/prune/summary.txt
files.download('results.zip')